In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, collect_set, first, max as spark_max, coalesce, when

vendor_name = "pipeline_consolidation"
layer_name = "silver_sites_resolved"

print("--- Starting Step 2c: Consolidation and Resolution ---")

# 1. Read from the entity resolution output table (post-GeoLink join)
df_resolved_input = spark.read.table("inlap.silver.sites_matched_registry")

required_columns = {
    "resolved_glid",
    "resolved_baseglid",
    "resolved_street_address",
    "resolved_unit_number",
    "resolved_city",
    "resolved_state",
    "resolved_zip",
    "resolved_location_type",
    "lat",
    "lon",
    "lat_rounded",
    "lon_rounded",
    "network_type",
    "status",
    "registry_confidence_score",
    "registry_last_verified",
    "source_vendor",
    "geolink_match_status",
}
missing_columns = sorted(required_columns - set(df_resolved_input.columns))
if missing_columns:
    raise ValueError(f"sites_matched_registry is missing required columns: {missing_columns}")

# 2. Filter for matched records (single_glid and ambiguous_multi_unit) that need consolidation
df_to_consolidate = df_resolved_input.filter(
    col("geolink_match_status").isin("single_glid", "ambiguous_multi_unit")
)

# 3. Group by resolved_glid and apply field-level precedence rules
df_consolidated = df_to_consolidate.groupBy("resolved_glid").agg(
    first("resolved_baseglid", ignorenulls=True).alias("resolved_baseglid"),
    first("resolved_street_address", ignorenulls=True).alias("street_address"),
    first("resolved_unit_number", ignorenulls=True).alias("unit_number"),
    first("resolved_city", ignorenulls=True).alias("city"),
    first("resolved_state", ignorenulls=True).alias("state"),
    first("resolved_zip", ignorenulls=True).alias("zip"),
    first("resolved_location_type", ignorenulls=True).alias("location_type"),
    first("lat", ignorenulls=True).alias("latitude"),
    first("lon", ignorenulls=True).alias("longitude"),
    first("lat_rounded", ignorenulls=True).alias("lat_rounded"),
    first("lon_rounded", ignorenulls=True).alias("lon_rounded"),
    first(
        when(col("source_vendor") == "here_tech", col("network_type")).otherwise(None),
        ignorenulls=True,
    ).alias("net_type_here"),
    first(
        when(col("source_vendor") == "opensignal", col("network_type")).otherwise(None),
        ignorenulls=True,
    ).alias("net_type_opensignal"),
    first(
        when(col("source_vendor") == "nv5", col("status")).otherwise(None),
        ignorenulls=True,
    ).alias("nv5_status_val"),
    spark_max("registry_confidence_score").alias("registry_confidence_score"),
    first("registry_last_verified", ignorenulls=True).alias("registry_last_verified"),
    collect_set("source_vendor").alias("contributing_vendors"),
)

# 4. Finalize precedence field selections and metadata timestamps
df_final_resolved = (
    df_consolidated.withColumn(
        "network_type",
        coalesce(col("net_type_here"), col("net_type_opensignal"), lit("UNKNOWN")),
    )
    .withColumn(
        "status",
        coalesce(col("nv5_status_val"), lit("Unknown")),
    )
    .withColumn("updated_timestamp", current_timestamp())
    .drop("net_type_here", "net_type_opensignal", "nv5_status_val")
)

# 5. Ensure all column names are strictly lowercase (safety check)
for c in df_final_resolved.columns:
    df_final_resolved = df_final_resolved.withColumnRenamed(c, c.lower())

# 6. Write consolidated output to silver_sites_resolved
df_final_resolved.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("inlap.silver.sites_resolved")

# 7. Log Audit Metrics
rows_input = df_to_consolidate.count()
rows_output = df_final_resolved.count()

spark.sql(f"""
    INSERT INTO inlap.control.audit_log 
    VALUES (
        '{vendor_name}', 
        '{layer_name}', 
        current_timestamp(), 
        'SUCCESS', 
        {rows_input}, 
        {rows_output}, 
        0
    )
""")

display(df_final_resolved.orderBy(col("resolved_glid")).limit(5))
print(f"Consolidation complete. Input matched rows: {rows_input} | Collapsed unique GLIDs: {rows_output}")